In [1]:
import random

import numpy as np
from matplotlib import pyplot as plt    

import optax
import equinox as eqx 
from jax import Array, numpy as jnp, random as jr, nn

from qwen import QwenModel, utils, forward

In [2]:
def load_dataset():
    loaded_data = np.load("data.npz")

    obs = loaded_data["obs"]
    actions = loaded_data["actions"]
    rewards = loaded_data["rewards"]

    obs_mean, obs_std = obs.mean(axis=0), obs.std(axis=0) + 1e-8
    obs = (obs - obs_mean) / obs_std

    actions = nn.one_hot(actions, num_classes=3)
    rewards = nn.one_hot(rewards + 1, num_classes=3)

    N = obs.shape[0]
    split = int(0.9 * N)

    train_obs, test_obs = obs[:split], obs[split:]
    train_actions, test_actions = actions[:split], actions[split:]
    train_rewards, test_rewards = rewards[:split], rewards[split:]

    return train_obs, train_actions, train_rewards, test_obs, test_actions, test_rewards

In [3]:
def preprocess(obs, actions):
    B, T, K, D = obs.shape
    obs_flat = jnp.reshape(obs, (B, T, K * D))
    return jnp.concatenate([obs_flat, actions], axis=-1)


def postprocess(out):
    B, T, D = out.shape
    obs, rewards = out[:, :, :-3], out[:, :, -3:]
    return jnp.reshape(obs, (B, T, 3, 4)), rewards


def train(
    key,
    model,
    optimizer,
    train_obs,
    train_actions,
    train_rewards,
    seq_len,
    batch_size,
    epochs,
    eval_callback=None,
):
    opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

    def loss_fn(model, prev_obs, prev_actions, next_obs, next_rewards):
        inputs = preprocess(prev_obs, prev_actions)
        ouputs = forward(model, inputs)
        pred_next_obs, pred_next_rewards = postprocess(ouputs)
        
        obs_loss = jnp.mean((pred_next_obs - next_obs) ** 2)
        reward_loss = jnp.mean((pred_next_rewards - next_rewards) ** 2)
        return obs_loss + reward_loss

    @eqx.filter_jit
    def step(model, opt_state, prev_obs, prev_actions, next_obs, next_rewards):
        loss_value, grads = eqx.filter_value_and_grad(loss_fn)(
            model, prev_obs, prev_actions, next_obs, next_rewards
        )
        updates, opt_state = optimizer.update(grads, opt_state, model)
        model = eqx.apply_updates(model, updates)
        return model, opt_state, loss_value

    for epoch in range(epochs):
        key, subkey = jr.split(key)
        idx = jr.randint(subkey, (batch_size,), 0, train_obs.shape[0] - seq_len)
        obs_batch = jnp.stack([train_obs[i : i + seq_len] for i in idx])
        act_batch = jnp.stack([train_actions[i : i + seq_len] for i in idx])
        rew_batch = jnp.stack([train_rewards[i : i + seq_len] for i in idx])
        prev_obs, next_obs = obs_batch[:, :-1], obs_batch[:, 1:]
        prev_act, next_rew = act_batch[:, :-1], rew_batch[:, 1:]
        model, opt_state, loss_value = step(
            model, opt_state, prev_obs, prev_act, next_obs, next_rew
        )

        if epoch % 100 == 0 and eval_callback is not None:
            eval_callback(model, epoch, loss_value)

    return model

In [4]:
@eqx.filter_jit
def generate(model, obs_context, actions, max_tokens):
    generated_obs = []
    generated_rewards = []
    current_obs = obs_context
    for i in range(max_tokens):
        cur_len = current_obs.shape[0]
        in_obs = current_obs[None, ...]
        in_act = actions[:cur_len][None, ...]
        x = preprocess(in_obs, in_act)
        y = forward(model, x)
        last_out = y[:, -1:, :]
        pred_obs, pred_rew = postprocess(last_out)
        generated_obs.append(pred_obs[0, 0])
        generated_rewards.append(pred_rew[0, 0])
        current_obs = jnp.concatenate([current_obs, pred_obs[0, 0:1]], axis=0)
    return jnp.stack(generated_obs, axis=0), jnp.stack(generated_rewards, axis=0)


def evaluate(model, test_obs, test_actions, test_rewards, context_len, max_tokens):
    start_idx = random.randint(0, test_obs.shape[0] - context_len - max_tokens)
    obs_context = test_obs[start_idx : start_idx + context_len]
    actions_seq = test_actions[start_idx : start_idx + context_len + max_tokens]

    true_obs = test_obs[start_idx + context_len : start_idx + context_len + max_tokens]
    true_rewards = test_rewards[
        start_idx + context_len : start_idx + context_len + max_tokens
    ]

    pred_obs, pred_rewards = generate(model, obs_context, actions_seq, max_tokens)

    fig, axes = plt.subplots(4, 4, figsize=(8, 8))

    for i in range(3):
        for j in range(4):
            axes[i, j].plot(pred_obs[:, i, j], color="red", label="Pred")
            axes[i, j].plot(true_obs[:, i, j], color="blue", label="True")
            axes[i, j].tick_params(axis="both", which="both", length=0)
            for spine in axes[i, j].spines.values():
                spine.set_linewidth(1.5)

    axes[3, 0].plot(pred_rewards[:, j], color="red", label="Pred")
    axes[3, 0].plot(true_rewards[:, j], color="blue", label="True")
    axes[3, 0].tick_params(axis="both", which="both", length=0)
    for spine in axes[3, 0].spines.values():
        spine.set_linewidth(1.5)
    axes[3, 0].legend(loc="center", frameon=False)

    plt.tight_layout()
    plt.show()

In [5]:
key = jr.PRNGKey(0)
train_obs, train_actions, train_rewards, test_obs, test_actions, test_rewards = load_dataset()

model = utils.init(
    key,
    input_dim=15,
    output_dim=15,
    hidden_size=128,
    num_layers=6,
    num_heads=4,
    num_key_value_heads=4,
    rope_theta=10000.0,
    rms_norm_eps=1e-6,
)

epochs = 10000
seq_len = 64
batch_size = 64
optimizer = optax.adamw(1e-4, weight_decay=0.01)

def eval_callback(model, epoch, loss):
    print(f"Epoch {epoch}: {loss}")
    evaluate(model, test_obs, test_actions, test_rewards, seq_len // 2,  seq_len // 2)


model = train(key, model, optimizer, train_obs, train_actions, train_rewards, seq_len, batch_size, epochs, eval_callback)